In [1]:
import pandas as pd

In [2]:
import sqlite3

In [3]:
conn = sqlite3.connect('data/hm.db') # 重新连接数据库

In [4]:
conn.execute("drop table if exists rfm_base")

In [5]:
conn.execute("""
create table rfm_base as 
select 
    customer_id,
    (julianday('2020-09-30') - julianday(max(last_date))) as recency_days,
    sum(active_days) as frequency,
    sum(amount) as monetary
from
    monthly_sales
group by 
    customer_id;
""")

In [6]:
rfm = pd.read_sql("select * from rfm_base limit 10", conn)

In [7]:
rfm.head(10)

,customer_id,recency_days,frequency,monetary
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,25.0,10,0.648983
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,84.0,23,2.601932
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,15.0,7,0.704780
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,479.0,1,0.060983
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,49.0,6,0.469695
5,000064249685c11552da43ef22a5030f35a147f723d5b0...,364.0,1,0.101644
6,0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...,16.0,3,0.166000
7,00007d2de826758b65a93dd24ce629ed66842531df6699...,140.0,16,3.823610
8,00007e8d4e54114b5b2a9b51586325a8d0fa74ea23ef77...,269.0,1,0.053356
9,00008469a21b50b3d147c97135e25b4201a8c58997f787...,688.0,1,0.078068


In [8]:
conn.execute("drop table if exists rfm_scored")

In [9]:
conn.execute("""
create table rfm_scored as 
select 
    customer_id, 
    recency_days,
    ntile(5) over (order by recency_days desc) as R,
    frequency,
    ntile(5) over (order by frequency asc) as F,
    monetary,
    ntile(5) over (order by monetary asc) as M
from rfm_base
""")

In [10]:
pd.read_sql("select * from rfm_scored limit 5", conn)

,customer_id,recency_days,R,frequency,F,monetary,M
0,8a2b646263ad3bf084c2e6a9fd0e7cab8445c7ea91df93...,741.0,1,1,1,0.002525,1
1,b01881934ced35e6fcaf2aa4fa87667e1ba1161ec6ffa9...,741.0,1,1,1,0.003373,1
2,8e43f1671cef0c6621d9629f7f60d920db8b397e7173e1...,741.0,1,1,1,0.005068,1
3,d3f8aa7b17fd03ba1a247192a3e4419b1f0b06bd246332...,741.0,1,1,1,0.005068,1
4,283082e0b6283ecbda0c0902a933975fb73c56134a9ceb...,741.0,1,1,1,0.005085,1


In [11]:
pd.read_sql("""
select customer_id, recency_days, R
from rfm_scored
where R =5
limit 10000
""", conn)

,customer_id,recency_days,R
0,d5798ee78ecd9565f0bbe2ef2caf55e05bfd6f0738130e...,42.0,5
1,e10e3d94d23bfc99437ef2310d37b0d06c031fb118807e...,42.0,5
2,808fb521e1e21ce606fec79099fb80710e11e53fb3577a...,42.0,5
3,3a0084faf0252e1681f9efe400c9a212bbc99d1769b30c...,42.0,5
4,6bd9441c63428ff04ca1a4dc00d18dba8cbd04442ae25a...,42.0,5
...,...,...,...
9995,c2224c334a7a65950157079b0f21d348fa862e5488679c...,41.0,5
9996,64db421a739257624ebc159cc2ac495de936298411bbe7...,41.0,5
9997,e4a40892ac6932d3060aaa15a71888ada195bc14cbce19...,41.0,5
9998,b42b9de12ebee79cbfc1cf8709e27d20102f6dbab6f3d2...,41.0,5


In [12]:
conn.execute("drop table if exists rfm_segment")

In [13]:
conn.execute("""
create table rfm_segment as 
select 
    *, -- 保留所有列
    case
        when R>=4 and F>=4 and M>=4 then '重要价值客户'
        when R>=4 and M>=4 then '重要发展客户' -- R, M high
        when F>=4 and M>=4 then '重要保持客户' -- F, M high
        when M>=4 then '重要挽留客户' -- M high
        when R>=4 and F>=4 then '一般价值客户' -- R F high
        when R>=4 then '一般发展客户'
        when F>=4 then '一般保持客户'
        else '流失客服'
    end as segment
from rfm_scored
""")

In [14]:
pd.read_sql("""
select segment, count(*)
from rfm_segment
group by segment
order by count(*) desc;
""", conn)

,segment,count(*)
0,流失客服,595126
1,重要价值客户,337152
2,一般发展客户,145779
3,重要保持客户,131296
4,重要挽留客户,55026
5,一般价值客户,40543
6,一般保持客户,35921
7,重要发展客户,21438


In [15]:
conn.execute("drop table if exists first_buy")

In [16]:
conn.execute("""
create table first_buy as
select 
    customer_id, 
    min(month) as cohort_month --客户首购月
from 
    monthly_sales
group by 
    customer_id;
""")

In [17]:
pd.read_sql("""
select *
from first_buy
order by customer_id
limit 10;
""", conn)

,customer_id,cohort_month
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,2018-12
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,2018-09
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2018-09
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,2019-06
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,2018-10
5,000064249685c11552da43ef22a5030f35a147f723d5b0...,2019-10
6,0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...,2019-06
7,00007d2de826758b65a93dd24ce629ed66842531df6699...,2018-09
8,00007e8d4e54114b5b2a9b51586325a8d0fa74ea23ef77...,2020-01
9,00008469a21b50b3d147c97135e25b4201a8c58997f787...,2018-11


In [18]:
conn.execute("""drop table if exists activity""")

In [19]:
conn.execute("""
create table activity as
select
    f.cohort_month, 
    m.month, 
    count(distinct m.customer_id) as active_users -- 数这个分组里有多少个客户
from monthly_sales as m
inner join first_buy as f on m.customer_id = f.customer_id
group by f.cohort_month, m.month; 
""")

In [20]:
pd.read_sql("""
select *
from activity
order by cohort_month
limit 1000
""", conn)

,cohort_month,month,active_users
0,2018-09,2018-09,140340
1,2018-09,2018-10,61551
2,2018-09,2018-11,59793
3,2018-09,2018-12,56736
4,2018-09,2019-01,53683
...,...,...,...
320,2020-07,2020-08,3198
321,2020-07,2020-09,1951
322,2020-08,2020-08,22974
323,2020-08,2020-09,2272


In [21]:
conn.execute("drop table if exists sizes")

In [22]:
conn.execute("""
create table sizes as 
select 
    cohort_month,
    count(*) as cohort_size -- 数这个分组有多少行，一行 = 一个客户
from first_buy
group by cohort_month
""")

In [23]:
pd.read_sql("""
select *
from sizes
limit 100
""", conn)

,cohort_month,cohort_size
0,2018-09,140340
1,2018-10,212669
2,2018-11,138233
3,2018-12,89944
4,2019-01,68957
5,2019-02,65453
6,2019-03,51185
7,2019-04,49884
8,2019-05,48645
9,2019-06,52127


In [24]:
conn.execute("drop table if exists cohort_retention")

In [27]:
conn.execute("""
create table cohort_retention as
select
    a.cohort_month, -- 哪批人
    a.month, -- 哪个月
    (cast(substr(a.month, 1, 4) as int) - cast(substr(a.cohort_month, 1, 4) as int))*12 + 
    cast(substr(a.month, 6, 2) as int) - cast(substr(a.cohort_month, 6, 2) as int) as month_n,
    -- 距首购过了几个月
    a.active_users, -- 分子
    s.cohort_size, -- 分母
    round(a.active_users * 100.0 / s.cohort_size, 2) as retention_pct
    -- 留存率 = active_users ÷ cohort_size × 100
from activity as a
inner join sizes as s on a.cohort_month = s.cohort_month
order by a.cohort_month, month_n
""")

In [30]:
pd.read_sql("""
select *
from cohort_retention
limit 25
""", conn)

,cohort_month,month,month_n,active_users,cohort_size,retention_pct
0,2018-09,2018-09,0,140340,140340,100.00
1,2018-09,2018-10,1,61551,140340,43.86
2,2018-09,2018-11,2,59793,140340,42.61
3,2018-09,2018-12,3,56736,140340,40.43
4,2018-09,2019-01,4,53683,140340,38.25
5,2018-09,2019-02,5,50781,140340,36.18
6,2018-09,2019-03,6,53249,140340,37.94
7,2018-09,2019-04,7,55296,140340,39.40
8,2018-09,2019-05,8,55332,140340,39.43
9,2018-09,2019-06,9,59931,140340,42.70


In [32]:
# 每个客户的"活跃月数 + 总消费"
pd.read_sql("""
select 
    customer_id,
    count(distinct month) as active_months,
    sum(amount) as total_amount
from monthly_sales
group by customer_id
""", conn)

,customer_id,active_months,total_amount
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,7,0.648983
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,12,2.601932
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,5,0.704780
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,1,0.060983
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,5,0.469695
...,...,...,...
1362276,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,9,1.292356
1362277,ffffcd5046a6143d29a04fb8c424ce494a76e5cdf4fab5...,15,1.807322
1362278,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,11,0.788932
1362279,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,2,0.209203


In [34]:
pd.read_sql("""
select 
    case 
        when active_months = 1 then '一次性购买' 
        else '复购客户'
    end as cust_type,
    count(*) as customer_count, -- 每类客户多少人
    count(*)*100.0 / sum(count(*)) over () as customer_pct, -- 人数占比（OVER ()在整个结果集上求和）
    sum(total_amount) as total_revenue, -- 每类收入
    sum(total_amount)*100.0 / sum(sum(total_amount)) over () as revenue_pct -- 收入占比
from (
    select 
        customer_id,
        count(distinct month) as active_months,
        sum(amount) as total_amount
    from monthly_sales
    group by customer_id
)
group by cust_type
""", conn)

,cust_type,customer_count,customer_pct,total_revenue,revenue_pct
0,一次性购买,491559,36.083525,50983.015610,5.763098
1,复购客户,870722,63.916475,833662.958441,94.236902
